In [1]:
import pandas as pd
import requests
import time  
import os

In [2]:
# 1. Verification Gate: Ensure the cleaned registry exists
registry_filename = "location_registry.csv"
if not os.path.exists(registry_filename):
    raise FileNotFoundError(f"Missing registry file! Please run Notebook 01A first.")

In [3]:
# Load our coordinates from the sanitized registry
registry_df = pd.read_csv(registry_filename)
print(f"Loaded {len(registry_df)} destinations successfully from the master registry.")

Loaded 10 destinations successfully from the master registry.


In [4]:
# 2. Define timeframe parameters and output filename
start_date = "2021-01-01"
end_date = "2025-12-31"
csv_filename = "tourist_locations_weather_raw.csv"

In [5]:
# Clear out old file versions to make sure we don't accidentally mix up data layers
if os.path.exists(csv_filename):
    os.remove(csv_filename)
    print(f"Removed old version of {csv_filename} to ensure a fresh clean pull.")

Removed old version of tourist_locations_weather_raw.csv to ensure a fresh clean pull.


In [6]:
# 3. Loop through your 10 cities using the values in our registry table
for index, row in registry_df.iterrows():
    city = row['location']
    lat = row['latitude']
    lon = row['longitude']
    
    url = (
        f"https://archive-api.open-meteo.com/v1/archive?"
        f"latitude={lat}&longitude={lon}&"
        f"start_date={start_date}&end_date={end_date}&"
        f"daily=temperature_2m_max,temperature_2m_min,precipitation_sum,rain_sum,snowfall_sum,wind_speed_10m_max&"
        f"timezone=auto"
    )
    
    success = False
    attempts = 0
    
    while not success and attempts < 5:
        print(f"Fetching data matrix for {city} [Lat: {lat}, Lon: {lon}]...")
        try:
            response = requests.get(url, timeout=15)
            
            if response.status_code == 200:
                data = response.json()["daily"]
                
                df_city = pd.DataFrame({
                    "location": city,
                    "date": data["time"],
                    "temp_max": data["temperature_2m_max"],
                    "temp_min": data["temperature_2m_min"],
                    "precipitation": data["precipitation_sum"],
                    "rain": data["rain_sum"],
                    "snowfall": data["snowfall_sum"],
                    "wind_speed": data["wind_speed_10m_max"]
                })
                
                if not os.path.exists(csv_filename):
                    df_city.to_csv(csv_filename, index=False)
                else:
                    df_city.to_csv(csv_filename, mode='a', header=False, index=False)
                    
                print(f"  -> [Success] Saved {len(df_city)} daily rows for {city}.\n")
                success = True
                
            elif response.status_code == 429:
                attempts += 1
                print(f"  -> [Rate Limit 429 Hit] Cooling down for 10 seconds (Attempt {attempts}/5)...")
                time.sleep(10)
            else:
                print(f"  -> API Error for {city}. Status code: {response.status_code}. Breaking loop.\n")
                break
                
        except Exception as e:
            print(f"  -> Connection Error occurred for {city}: {e}. Retrying after 5 seconds...")
            time.sleep(5)
            attempts += 1
            
    # Base polite spacing delay between successful city requests
    time.sleep(1.5)

Fetching data matrix for Manali [Lat: 32.2396, Lon: 77.1887]...
  -> [Success] Saved 1826 daily rows for Manali.

Fetching data matrix for Shimla [Lat: 31.1048, Lon: 77.1734]...
  -> [Success] Saved 1826 daily rows for Shimla.

Fetching data matrix for Mussoorie [Lat: 30.4598, Lon: 78.0644]...
  -> [Success] Saved 1826 daily rows for Mussoorie.

Fetching data matrix for Nainital [Lat: 29.3802, Lon: 79.4636]...
  -> [Success] Saved 1826 daily rows for Nainital.

Fetching data matrix for Leh [Lat: 34.1526, Lon: 77.5771]...
  -> [Success] Saved 1826 daily rows for Leh.

Fetching data matrix for Darjeeling [Lat: 27.041, Lon: 88.2627]...
  -> [Success] Saved 1826 daily rows for Darjeeling.

Fetching data matrix for Goa [Lat: 15.2993, Lon: 74.124]...
  -> [Success] Saved 1826 daily rows for Goa.

Fetching data matrix for Jaipur [Lat: 26.9124, Lon: 75.7873]...
  -> [Success] Saved 1826 daily rows for Jaipur.

Fetching data matrix for Munnar [Lat: 10.0889, Lon: 77.0595]...
  -> [Success] Saved

In [7]:
print("=====================================================================")
print("🎉 TASK 01B WEATHER HARVESTING PROCESS COMPLETE")
print("=====================================================================\n")

# 4. Final verification read-back
verify_df = pd.read_csv(csv_filename)
print(f"Total Rows Captured across all cities: {len(verify_df)}")
print("\nUnique Date Rows per Destination Registry Item:")
print(verify_df["location"].value_counts())

🎉 TASK 01B WEATHER HARVESTING PROCESS COMPLETE

Total Rows Captured across all cities: 18260

Unique Date Rows per Destination Registry Item:
location
Manali        1826
Shimla        1826
Mussoorie     1826
Nainital      1826
Leh           1826
Darjeeling    1826
Goa           1826
Jaipur        1826
Munnar        1826
Ooty          1826
Name: count, dtype: int64


In [8]:
# Duplicate Check
duplicates = verify_df.duplicated(
    subset=["location", "date"]
).sum()

print("Duplicates:", duplicates)

Duplicates: 0


In [9]:
# Missing Values Check
print(verify_df.isnull().sum())

location         0
date             0
temp_max         0
temp_min         0
precipitation    0
rain             0
snowfall         0
wind_speed       0
dtype: int64


In [10]:
verify_df = verify_df.bfill()

In [11]:
# Date Coverage Check
print(verify_df["date"].min())
print(verify_df["date"].max())

2021-01-01
2025-12-31


In [12]:
# Weather Sanity Check
print(verify_df.describe())

           temp_max      temp_min  precipitation          rain      snowfall  \
count  18260.000000  18260.000000   18260.000000  18260.000000  18260.000000   
mean      20.258182     11.593582       5.613198      5.165361      0.313712   
std        8.410027      8.304102      12.777172     12.389725      2.339933   
min      -12.600000    -21.700000       0.000000      0.000000      0.000000   
25%       15.900000      6.600000       0.000000      0.000000      0.000000   
50%       20.400000     12.600000       0.700000      0.500000      0.000000   
75%       23.900000     16.200000       5.500000      4.800000      0.000000   
max       46.500000     33.000000     293.400000    293.400000    104.650000   

         wind_speed  
count  18260.000000  
mean      10.454704  
std        4.456872  
min        2.500000  
25%        7.100000  
50%        9.400000  
75%       12.800000  
max       42.900000  


In [13]:
# Create derived weather features now and save them.
verify_df["temperature_range"] = (
    verify_df["temp_max"] - verify_df["temp_min"]
)

verify_df["heavy_rain_flag"] = (
    verify_df["rain"] >= 50
).astype(int)

verify_df["snow_flag"] = (
    verify_df["snowfall"] > 0
).astype(int)

verify_df["strong_wind_flag"] = (
    verify_df["wind_speed"] >= 25
).astype(int)

In [14]:
verify_df.to_csv("weather_features.csv", index = False)

In [15]:
 verify_df

,location,date,temp_max,temp_min,precipitation,rain,snowfall,wind_speed,temperature_range,heavy_rain_flag,snow_flag,strong_wind_flag
0,Manali,2021-01-01,9.4,4.9,0.0,0.0,0.00,4.7,4.5,0,0,0
1,Manali,2021-01-02,7.8,5.7,6.4,0.0,4.48,4.0,2.1,0,1,0
2,Manali,2021-01-03,8.7,5.6,18.4,0.0,12.88,4.8,3.1,0,1,0
3,Manali,2021-01-04,8.9,5.6,4.0,0.0,2.80,5.4,3.3,0,1,0
4,Manali,2021-01-05,7.2,6.0,48.5,0.2,33.81,3.8,1.2,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...
18255,Ooty,2025-12-27,18.1,7.7,0.0,0.0,0.00,11.0,10.4,0,0,0
18256,Ooty,2025-12-28,18.5,6.9,0.0,0.0,0.00,11.4,11.6,0,0,0
18257,Ooty,2025-12-29,18.7,8.5,0.0,0.0,0.00,8.7,10.2,0,0,0
18258,Ooty,2025-12-30,19.0,9.1,0.4,0.4,0.00,12.4,9.9,0,0,0
